<a href="https://colab.research.google.com/github/wendysbustillo-sketch/SIMULACION-DE-SISTEMAS/blob/main/Practica3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install simpy
import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} inspeccionada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque))
env.run(until=500)  # simular 500 segundos

Botella 2 moldeada en 26.5
Botella 1 moldeada en 34.5
Botella 3 moldeada en 80.5
Botella 2 enfriada en 86.5
Botella 1 enfriada en 94.5
Botella 4 moldeada en 103.3
Botella 3 enfriada en 140.5
Botella 2 inspeccionada en 154.3
Botella 4 enfriada en 163.3
Botella 5 moldeada en 202.5
Botella 6 moldeada en 205.4
Botella 1 inspeccionada en 221.0
Botella 3 inspeccionada en 240.9
Botella 4 inspeccionada en 251.1
Botella 5 enfriada en 262.5
Botella 6 enfriada en 265.4
Botella 7 moldeada en 300.4
Botella 8 moldeada en 302.8
Botella 5 inspeccionada en 308.4
Botella 6 inspeccionada en 324.5
Botella 7 enfriada en 360.4
Botella 8 enfriada en 362.8
Botella 10 moldeada en 370.9
Botella 9 moldeada en 373.0
Botella 7 inspeccionada en 374.6
Botella 11 moldeada en 403.9
Botella 12 moldeada en 404.3
Botella 13 moldeada en 420.8
Botella 10 enfriada en 430.9
Botella 9 enfriada en 433.0
Botella 11 enfriada en 463.9
Botella 12 enfriada en 464.3
Botella 8 inspeccionada en 464.7
Botella 14 moldeada en 465.7
Botel